# 08 — Dual-Attention Transformer–CNN–BiLSTM

**Family:** CNN–BiLSTM–Transformer &nbsp;•&nbsp; **Input:** raw signal `[14 x window_samples]`

A hybrid sequence model with **two explicit attention mechanisms**:

1. **Channel attention** — squeeze-and-excitation gating over the 14 electrodes.
2. **Temporal attention** — additive attention pooling over time steps.

Pipeline: channel attention → temporal CNN feature extractor → BiLSTM → Transformer
encoder block → temporal-attention pooling → linear head.

## 1. Setup

In [ ]:
import os, sys

import torch

# make the `src` package importable when the notebook runs from notebooks/
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src import benchmark_si as B
from src.evaluation import set_seed, get_device, count_parameters
from src.data_loader import load_subject
from src.models import dual_attention

set_seed(42)
DEVICE = get_device()
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
LOSO_SPLITS = os.path.join(ROOT, "data", "splits", "loso_splits.json")
RESULTS_DIR = os.path.join(ROOT, "results")

# ----- experiment configuration -----
MODEL_NAME = "DualAttention_TransformerCNNBiLSTM"
LABEL = "valence"     # main comparison target. Swap to "arousal" / "dominance" to reproduce.
QUICK = False         # True  -> 3-fold / few-epoch smoke test
                      # False -> the full 23-fold benchmark used for the thesis table
print("device:", DEVICE, "| model:", MODEL_NAME, "| label:", LABEL, "| QUICK:", QUICK)
print("recipe:", B.REGISTRY[MODEL_NAME])


## 2. Architecture

`src/models/dual_attention.py`: `ChannelAttention` (SE, reduction 2) → 2 conv/pool
blocks (`Conv1d`, time ÷4) → single-layer **BiLSTM** (hidden 64 each direction) →
1 Transformer encoder layer (4 heads) → `TemporalAttentionPool` (Bahdanau-style) →
LayerNorm + linear. The learned channel-attention weights are cached on
`model._last_channel_weights` for interpretability plots.

In [ ]:
# Instantiate once on a dummy spec just to print the graph + parameter count.
_dummy_meta = {"input_shape": (14, load_subject(PROCESSED_DIR, "subject_01").window_samples)}
_m = dual_attention.build({"n_classes": 2, "meta": _dummy_meta})
print(_m)
print(f"\ntrainable parameters: {count_parameters(_m):,}")
del _m

## Validation protocol — subject-independent (LOSO)

The benchmark uses a single protocol so every architecture is scored identically:
**23-fold leave-one-subject-out** (`data/splits/loso_splits.json`). Fold *k* trains a
fresh model on 22 subjects and tests on the held-out 23rd; normalisation statistics
are fit on the training subjects only and a 10% slice of them is used for early
stopping. We report the **mean ± std of the headline metrics across the 23 folds**.

Everything runs through `src.benchmark_si.run_one` — the one entry point shared by
every model notebook — which reads this model's recipe from `benchmark_si.REGISTRY`,
trains through `src.evaluation.run_loso`, and writes `results/{MODEL_NAME}_results.json`.


## 3. Run the subject-independent (LOSO) benchmark

In [ ]:
# One shared call: 23-fold LOSO for this model, then writes results/DualAttention_TransformerCNNBiLSTM_results.json
res = B.run_one(MODEL_NAME, label=LABEL, quick=QUICK, device=DEVICE)
res

## 4. Channel-attention profile (interpretability)

Averaging the SE gate over a held-out fold shows which electrodes the dual-attention
model weights most for the emotion decision — a compact companion figure to DGCNN's
learned adjacency. Below we train one quick model on LOSO fold 0 (22 subjects).

In [ ]:
import matplotlib.pyplot as plt

from src.data_loader import build_loso_loaders
from src.evaluation import make_default_fit
from src.models.dual_attention import DualAttentionNet

tl, vl, el, meta = build_loso_loaders(
    PROCESSED_DIR, LOSO_SPLITS, fold=0,
    label=LABEL, feature="raw", batch_size=128, normalize=True, val_frac=0.0,
)
m = DualAttentionNet(n_channels=meta["input_shape"][0],
                     n_samples=meta["input_shape"][1], n_classes=2).to(DEVICE)
_ = make_default_fit(epochs=(5 if QUICK else 40), lr=7e-4)(m, tl, None, DEVICE)

m.eval(); ws = []
with torch.no_grad():
    for xb, _ in el:
        m(xb.to(DEVICE)); ws.append(m._last_channel_weights.cpu())
w = torch.cat(ws).mean(0).numpy()
chans = load_subject(PROCESSED_DIR, meta["test_subject"]).channel_names

plt.figure(figsize=(7, 3))
plt.bar(chans, w); plt.ylabel("mean SE gate"); plt.xticks(rotation=90)
plt.title("Dual-Attention channel importance (LOSO fold 0)")
plt.tight_layout()
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, "figures", "dualattn_channel_importance_loso_fold0.png"), dpi=150)
plt.show()

## Notes for the methodology chapter

* Every metric comes from `src/evaluation.compute_metrics` (accuracy, macro
  precision / recall / F1, confusion matrix) — no per-notebook metric code.
* The per-model LOSO recipe (feature view, batch size, epochs, lr) lives in
  `src/benchmark_si.py::REGISTRY` — a single source of truth shared by this
  notebook and the `python -m src.benchmark_si` CLI.
* `params_count` and `train_time_sec` are logged to
  `results/{MODEL_NAME}_results.json` for the accuracy-vs-efficiency analysis.
* Set `QUICK = True` to smoke-test the whole flow (3 folds / few epochs); `LABEL`
  switches the target between valence / arousal / dominance with no other change.
